In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# 1. Device and Seed Configuration
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

batch_size = 128
num_epochs = 10  # Standardized for fair comparison

# Base transformations for standardized evaluation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=transform), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(datasets.CIFAR10(root='./data', train=False, download=True, transform=transform), batch_size=batch_size, shuffle=False)

# =====================================================================
# 2. Reference Models (From Your Previous Tasks)
# =====================================================================
class InitialMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class InitialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        x = F.max_pool2d(torch.sigmoid(self.conv1(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv2(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv3(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv4(x)), 2)
        x = x.view(x.size(0), -1)
        return self.fc2(torch.sigmoid(self.fc1(x)))

# =====================================================================
# 3. CREATIVITY TASK: Dual-Branch CNN-MLP Hybrid Network
# =====================================================================
class DualBranchHybridNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # --- CNN Branch (Extracts local features/patterns) ---
        self.cnn_conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.cnn_conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.cnn_fc = nn.Linear(64 * 8 * 8, 128) # Feature vector from CNN
        
        # --- MLP Branch (Captures global spatial context) ---
        self.mlp_fc1 = nn.Linear(32 * 32 * 3, 256)
        self.mlp_fc2 = nn.Linear(256, 128) # Feature vector from MLP
        
        # --- Fusion Classifier ---
        # Concatenated feature size: 128 (CNN) + 128 (MLP) = 256
        self.final_fc = nn.Linear(128 + 128, 10)

    def forward(self, x):
        # 1. Process CNN Branch
        c = F.max_pool2d(torch.relu(self.cnn_conv1(x)), 2)
        c = F.max_pool2d(torch.relu(self.cnn_conv2(c)), 2)
        c = c.view(c.size(0), -1)
        cnn_features = torch.relu(self.cnn_fc(c))
        
        # 2. Process MLP Branch
        m = x.view(x.size(0), -1)
        m = torch.relu(self.mlp_fc1(m))
        mlp_features = torch.relu(self.mlp_fc2(m))
        
        # 3. Fusion via Concatenation
        fused_features = torch.cat((cnn_features, mlp_features), dim=1)
        
        # 4. Final Classification
        return self.final_fc(fused_features)

# =====================================================================
# 4. Universal Pipeline Function
# =====================================================================
def run_pipeline(model_class, name):
    print(f"\n--- Training {name} ---")
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    history = {'train_loss': [], 'test_acc': []}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                _, predicted = model(images).max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total
        history['train_loss'].append(epoch_loss)
        history['test_acc'].append(epoch_acc)
        print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.4f} | Test Acc: {epoch_acc:.2f}%")
        
    return history

# =====================================================================
# 5. Execution and Plotting
# =====================================================================
results = {
    'Standard MLP': run_pipeline(InitialMLP, 'Standard MLP'),
    'Standard CNN (Sigmoid)': run_pipeline(InitialCNN, 'Standard CNN (Sigmoid)'),
    'Our Hybrid Network': run_pipeline(DualBranchHybridNet, 'Our Hybrid Network')
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, metrics in results.items():
    axes[0].plot(range(1, num_epochs + 1), metrics['train_loss'], label=name, marker='o')
axes[0].set_title('Training Loss Convergence')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True)

for name, metrics in results.items():
    axes[1].plot(range(1, num_epochs + 1), metrics['test_acc'], label=name, marker='s')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()